In [2]:
# app.py
from time import perf_counter  # 新增：计时函数

from modeling import build_models_from_csv
from bundles import BaseBundle, MSBundle
from utils import (
    tighten_bounds_one_model,
    MIN_DIST, ACTIVE_TOL, GAP_STOP_TOL,
)
from simplex import run_pid_simplex_3d

RUN_QUICK_TEST = True  # True: 先用小规模验证

if RUN_QUICK_TEST:
    csv_path       = "data.csv"
    max_scenarios  = 2
    target_nodes   = 13
else:
    csv_path       = "data.csv"
    max_scenarios  = 99
    target_nodes   = 30

bounds = {
    "x":  (None, None),
    "u":  (None, None),
    "e":  (None, None),
    "I":  (-10, 10),
    "Kp": (0, 1),
    "Ki": (0, 1),
    "Kd": (0, 1),
}
weights = (1.0, 0.01)

# ====== 阶段 1：数据加载与场景构造 ======
t_load0 = perf_counter()
model_list, first_stg_vars_list, m_tmpl_list, T = build_models_from_csv(
    csv_path, h=0.2, weights=weights, bounds=bounds,
    sp0=0.0, sp1=0.5, ku_col="tau_us", tau_col="tau_xs",
    disturb_prefix="disturbance_", setpoint_change_col="setpoint_change",
    max_scenarios=max_scenarios, skip=0
)
t_load1 = perf_counter()
print(f"[Time] Data load & scenario build: {t_load1 - t_load0:.3f}s")

# ====== 阶段 1.5：FBBT / OBBT（在持久化实例化之前进行）======
# 对齐他们 PID 脚本的默认：FBBT 开、OBBT 开，OBBT 用轻量选项
obbt_solver_opts = {
    "NonConvex": 2,
    "MIPGap": 1,     # 宽松
    "TimeLimit": 5   # 轻量
}
for m, yvars in zip(model_list, first_stg_vars_list):
    tighten_bounds_one_model(m, yvars,
                             use_fbbt=True,
                             use_obbt=True,
                             obbt_solver_name="gurobi",
                             obbt_solver_opts=obbt_solver_opts,
                             max_rounds=3, tol=1e-6, verbose=True)

# ====== 阶段 2：求解器持久化包装 ======
ub_options = {
    'NonConvex': 2,           # 他们的 UB 只强调非凸
    # 按需可再加 TimeLimit 等；他们脚本里 UB 不怎么加别的
}
lb_options = {
    'NonConvex': 2,
    'MIPGap': 0.2,            # 他们 LB 初始较宽
    'TimeLimit': 15           # 他们 LB 设了时间上限
}

t_wrap0 = perf_counter()
base_bundles = [BaseBundle(m, ub_options) for m in model_list]  # UB 侧
ms_bundles   = [MSBundle(m, yvars, lb_options) for m, yvars in zip(model_list, first_stg_vars_list)]  # LB 侧
t_wrap1 = perf_counter()
print(f"[Time] Persistent wrapper (GurobiPersistent) setup: {t_wrap1 - t_wrap0:.3f}s")

# ====== stage 3：main loop ======
agg_bundle = None  # 对每个场景分别算，不启用共享 λ 聚合

t_run0 = perf_counter()
hist = run_pid_simplex_3d(
    base_bundles=base_bundles,
    ms_bundles=ms_bundles,
    model_list=model_list,
    first_vars_list=first_stg_vars_list,
    target_nodes=target_nodes,
    min_dist=MIN_DIST,
    active_tol=ACTIVE_TOL,
    verbose=True,
    agg_bundle=agg_bundle,
    gap_stop_tol=1e-1,   # <== 例如改成 1e-5；或传 None 禁用
)
t_run1 = perf_counter()
print(f"[Time] Main loop total: {t_run1 - t_run0:.3f}s")

# ====== 结果输出 ======
print("\n==== Done ====")
print(f"Total nodes: {len(hist['nodes'])}")
print(f"Best UB: {min(hist['UB_hist']) if hist['UB_hist'] else None}")
print(f"Last LB: {hist['LB_hist'][-1] if hist['LB_hist'] else None}")

[Time] Data load & scenario build: 0.004s
[Tighten] rounds=1, changed=False
[Tighten] rounds=1, changed=False
Set parameter MIPGap to value 0.1
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter MIPGap to value 0.1
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter MIPGap to value 0.2
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter TimeLimit to value 15
Set parameter MIPGap to value 0.2
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter TimeLimit to value 15
[Time] Persistent wrapper (GurobiPersistent) setup: 0.037s
[Iter 0] created=6 (cum=6), active=6, active+UB=2, ms_recomputed=6
[Iter 0] Optimality gap: 3.440502e-01 (54.349%)
[Iter 0] Active simplex ratio = 1.000000
[Iter 0] UB node (1.0, 1.0, 1.0) is in 

[Iter 0] Elapsed: 15.970s
[Iter 1] created=12 (cum=18), active=10, active+UB=4, ms_recomputed=12
[Iter 1] Optimality gap: 1.656235e+00 (261.632%)
[Iter 1] Active simplex ratio = 1.000000
[Iter 1] UB node (1.0, 1.0, 1.0) is in simplices [2, 3, 6, 7, 10, 11]
[Iter 1] LB = -1.023195 = UB(0.633040) + ms_b(-1.656e+00) from T10
[Iter 1] LB = -1.023195 = UB(0.633040) + ms_b(-1.656e+00) from T10
[Iter 1] candidate rank #1: T10, scene=0, ms=-9.199e-01
== ms candidates (sorted by (ms, -dist)) ==
rank   simp   scene           ms    mind(all)                             pt
----------------------------------------------------------------------------
   1    T10       0  -9.1991e-01     4.41e-01       (0.3117, 0.3117, 1.0000)
   2    T11       0  -9.1991e-01     4.41e-01       (0.3117, 0.3117, 1.0000)
   3    T10       1  -7.3633e-01     4.04e-01       (0.2857, 0.2857, 1.0000)
   4    T11       1  -7.3633e-01     4.04e-01       (0.2857, 0.2857, 1.0000)
   5     T6       1  -4.8821e-01     3.95e-01  

[Iter 1] Elapsed: 32.247s
[Iter 2] created=18 (cum=36), active=13, active+UB=4, ms_recomputed=12
[Iter 2] Optimality gap: 7.781787e-01 (122.927%)
[Iter 2] Active simplex ratio = 0.957576
[Iter 2] UB node (1.0, 1.0, 1.0) is in simplices [2, 3, 6, 7, 10, 11, 14, 15]
[Iter 2] LB = -0.145138 = UB(0.633040) + ms_b(-7.782e-01) from T10
[Iter 2] LB = -0.145138 = UB(0.633040) + ms_b(-7.782e-01) from T10
[Iter 2] candidate rank #1: T10, scene=1, ms=-4.882e-01
== ms candidates (sorted by (ms, -dist)) ==
rank   simp   scene           ms    mind(all)                             pt
----------------------------------------------------------------------------
   1    T10       1  -4.8821e-01     3.95e-01       (0.2791, 1.0000, 0.2791)
   2    T11       1  -4.8821e-01     3.95e-01       (0.2791, 1.0000, 0.2791)
   3    T11       0  -2.8997e-01     4.54e-01       (0.3214, 1.0000, 0.3214)
   4    T10       0  -2.8997e-01     4.54e-01       (0.3214, 1.0000, 0.3214)
   5     T3       0  -1.3590e-01     3.

[Iter 2] Elapsed: 28.198s
[Iter 3] created=23 (cum=59), active=15, active+UB=4, ms_recomputed=14
[Iter 3] Optimality gap: 2.524776e-01 (39.883%)
[Iter 3] Active simplex ratio = 0.913145
[Iter 3] UB node (1.0, 1.0, 1.0) is in simplices [0, 5, 8, 9, 11, 12, 15, 16, 21, 22]
[Iter 3] LB = 0.380563 = UB(0.633040) + ms_b(-2.525e-01) from T9
[Iter 3] LB = 0.380563 = UB(0.633040) + ms_b(-2.525e-01) from T9
[Iter 3] candidate rank #1: T9, scene=0, ms=-1.449e-01
== ms candidates (sorted by (ms, -dist)) ==
rank   simp   scene           ms    mind(all)                             pt
----------------------------------------------------------------------------
   1     T9       0  -1.4492e-01     4.05e-01       (0.4143, 0.6354, 0.7789)
   2     T8       0  -1.4492e-01     4.05e-01       (0.4143, 0.6354, 0.7789)
   3     T5       0  -1.3590e-01     3.67e-01       (0.5709, 0.5709, 1.0000)
   4     T9       1  -1.0756e-01     4.75e-01       (0.3503, 0.6649, 0.6855)
   5     T8       1  -1.0756e-01     

[Iter 3] Elapsed: 37.200s
[Iter 4] created=30 (cum=89), active=20, active+UB=6, ms_recomputed=16
[Iter 4] Optimality gap: 2.186071e-01 (34.533%)
[Iter 4] Active simplex ratio = 0.870151
[Iter 4] UB node (1.0, 1.0, 1.0) is in simplices [0, 1, 2, 4, 10, 11, 20, 21, 24, 25, 28, 29]
[Iter 4] LB = 0.414433 = UB(0.633040) + ms_b(-2.186e-01) from T10
[Iter 4] LB = 0.414433 = UB(0.633040) + ms_b(-2.186e-01) from T10
[Iter 4] candidate rank #1: T10, scene=0, ms=-1.359e-01
== ms candidates (sorted by (ms, -dist)) ==
rank   simp   scene           ms    mind(all)                             pt
----------------------------------------------------------------------------
   1    T10       0  -1.3590e-01     2.78e-01       (0.5709, 0.5709, 1.0000)
   2    T11       0  -1.3590e-01     2.78e-01       (0.5709, 0.5709, 1.0000)
   3     T0       0  -1.1924e-01     4.02e-01       (1.0000, 0.4023, 1.0000)
   4    T10       1  -8.2702e-02     3.25e-01       (0.6267, 0.5281, 1.0000)
   5    T11       1  -8.23

[Iter 4] Elapsed: 41.230s
[Time] Main loop total: 154.928s

==== Done ====
Total nodes: 13
Best UB: 0.6330403478608069
Last LB: 0.4144332739022512
